# Diagonalisation exacte 

## But de l'algorithme

On cherche à réaliser la projection d'un hamiltonien H sur un espace de Krylov de taille L. On obtiendra une matrice T tridiagonale.

Pourquoi faire cela ? Il s'avère que les valeurs propres de T (surtout celles de module le plus haut) convergent rapidement vers celles de H. Ainsi, on a d'excellentes approximations pour $E_0$.

## Les modules à importer 

In [1]:
import numpy as np # pour les matrices denses 
import scipy.sparse as sp # pour les matrices sparse (creuses)

## L'algorithme de Lanczos 

Données nécessaires pour la méthode : 
- $init$ : il s'agit du vecteur à partir duquel nous allons construire le sous-espace de Krylov.
- $H$ : le hamiltonien. La méthode fonctionne pour des matrices de type sparse ou numpy. Ceci étant dit, il est conseillé d'utiliser le type sparse pour des raisons d'optimalité de temps de calcul.
- $L$ : nombre d'itérations maximal. Pour un rapport précision/temps de calcul adapté, nous vous conseillons de ne pas excéder $L=100$.
- $eps$ : la précision attendue. Nous recommandons pour déterminer $eps$ d'utiliser la relation de Physical Review B99, de Arata TANAKA : $eps=\alpha_{lan}\sqrt{N_{sys}}\textbf{u}$, où $\alpha_{lan}=10$, $N_{sys}$ la taille du hamiltonien, et $\textbf{u}$ est le epsilon de la machine, prenez $\textbf{u}=10^{-15}$.

In [2]:
def lanczos(init, H, L, eps):
    a = []
    b = []
    v = init
    b0 = np.linalg.norm(v)
    b.append(b0)
    v = v/b0
    w = np.zeros_like(v)
    w += H@v
    a.append((v.T)@w)
    w = w - a[0]*v
    b.append(np.linalg.norm(w))
    for i in range(1,L):
        if b[-1]<eps : break
        w = w/b[-1]
        v = -v*b[-1]
        v, w = w, v
        w = w + H@v
        a.append(v.T@w)
        w = w - a[-1]*v
        b.append(np.linalg.norm(w))
    return a, b[:-1]

L'algorithme renvoie deux arrays.

Le premier array représente les coefficients diagonaux de T (la matrice tridiagonale) et le second array les coefficients sur la diagonale en dessous et au dessous de la principale.

## Fonction de Green

L'avantage de l'algorithme de Lanczos est qu'il permet de déterminer une valeur approchée de la transformée de Fourier de la fonction de Green. 

On a le résultat \begin{align}
  G(z)&=\langle \psi \lvert \frac{1}{z-H}\lvert \psi \rangle \\
   &=  \frac{\langle \psi \lvert \psi \rangle}{z-a_0-\frac{b_1^2}{z-a_1-\frac{b_2^2}{z-a_2-\frac{b_3^2}{...}}}}
   &=  \frac{1}{z-a_0-\frac{b_1^2}{z-a_1-\frac{b_2^2}{z-a_2-\frac{b_3^2}{...}}}}
\end{align}

où $z$ est un nombre complexe.

En reprenant les coefficients a et b précédement calculés :

In [4]:
def greenLanczosAdapt(z, a, b):
    frac = 0
    n = len(b)
    for i in range(n-1, 0, -1):
        frac = b[i]**2/(z-a[i]-frac)
    return 1/(z-a[0]-frac)

## Fonction spectrale

En notant $A$ la fonction spectrale, nous avons la relation liant $A$ et $G$ la fonction de Green complexe :

\begin{align}
A(\omega \pm i\eta)=\mp \frac{1}{\pi} \Im\left(G(\omega \pm i\eta)\right)
\end{align}

En général, on choisit $\eta \sim \frac{1}{L}$

In [5]:
def greenLanczos(omega, mu, init, H, L):
    a, b = lanczos(init, H, L)
    H_chap = np.diag(a) + np.diag(b[1:], k=-1) + np.diag(b[1:], k=1) # reconstruction du hamiltonien tridiagonal
    m_chap = np.linalg.eigvals(H_chap) # calcul de ses valeurs propres 
    E_0 = m_chap[0]
    return greenLanczosAdapt(complex(omega-E_0, mu), a, b) 